# Train Vibration Analysis – Complete Notebook
This notebook processes **multiple binary acceleration files** (≈20 files).

Pipeline:
1. Load binary data
2. Preprocess (offset removal + bandpass filter)
3. Segment train passage
4. Extract features
5. Cluster train passages & estimate speed proxy


## 1. Imports and Global Parameters

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import scipy.signal as sig
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

FS = 2000.0  # Sampling frequency (Hz)

## 2. Load Binary Signal

In [ ]:
def read_trace(path):
    return np.fromfile(path, dtype=np.float32)

## 3. Preprocessing

In [ ]:
def preprocess(x, fs=FS):
    x = x - np.median(x)
    sos = sig.butter(4, [5, 400], btype='bandpass', fs=fs, output='sos')
    return sig.sosfiltfilt(sos, x)

## 4. Segment Train Passage

In [ ]:
def segment_passage(x, fs=FS, win_s=0.1):
    win = int(win_s * fs)
    rms = np.sqrt(sig.convolve(x**2, np.ones(win)/win, mode='same'))
    med = np.median(rms)
    mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + 6 * mad
    idx = np.where(rms > thr)[0]
    if len(idx) == 0:
        return x
    splits = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[splits + 1]]
    ends = np.r_[idx[splits], idx[-1]]
    k = np.argmax(ends - starts)
    s, e = starts[k], ends[k]
    pad = int(0.5 * fs)
    return x[max(0,s-pad):min(len(x), e+pad)]

## 5. Feature Extraction

In [ ]:
def extract_features(x, fs=FS):
    f, Pxx = sig.welch(x, fs=fs, nperseg=min(4096, len(x)))
    env = np.abs(sig.hilbert(x))
    fe, Pe = sig.welch(env, fs=fs, nperseg=min(4096, len(env)))
    dom_env_freq = fe[(fe>=1)&(fe<=30)][np.argmax(Pe[(fe>=1)&(fe<=30)])]
    return {
        'duration_s': len(x)/fs,
        'rms': np.sqrt(np.mean(x**2)),
        'peak': np.max(np.abs(x)),
        'spec_centroid': np.sum(f*Pxx)/np.sum(Pxx),
        'env_dom_freq': dom_env_freq
    }

## 6. Run Pipeline on All Files

In [ ]:
DATA_DIR = '.'  # change if needed
files = glob.glob(os.path.join(DATA_DIR, '*.bin'))
rows = []
for fpath in files:
    raw = read_trace(fpath)
    x = preprocess(raw)
    seg = segment_passage(x)
    feats = extract_features(seg)
    feats['file'] = os.path.basename(fpath)
    rows.append(feats)

df = pd.DataFrame(rows)
df

## 7. Clustering (Train Types)

In [ ]:
X = df.drop(columns=['file']).values
Xs = StandardScaler().fit_transform(X)
Z = PCA(n_components=2).fit_transform(Xs)

kmeans = KMeans(n_clusters=min(4, len(df)), random_state=0, n_init='auto')
df['cluster'] = kmeans.fit_predict(Z)

plt.scatter(Z[:,0], Z[:,1], c=df['cluster'])
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('Train Passage Clusters')
plt.show()

df

## 8. Save Results

In [ ]:
df.to_csv('train_analysis_results.csv', index=False)
print('Saved train_analysis_results.csv')